In [ ]:
# Load the aligned crop cover
crop_cover_aligned = rioxarray.open_rasterio(crop_cover_aligned_path, chunks="auto")
if 'band' in crop_cover_aligned.dims:
    crop_cover_aligned = crop_cover_aligned.squeeze('band', drop=True)

print(f"Aligned crop_cover shape: {crop_cover_aligned.shape}")
print(f"Woreda_grid shape: {woreda_grid.shape}")
print(f"Shapes match: {crop_cover_aligned.shape == woreda_grid.shape}")

# Now we can mask woreda_grid by crop cover
# Keep only woreda pixels where there is crop cover (value = 2)
crop_mask = (crop_cover_aligned == 2)
masked_woredas = woreda_grid.where(crop_mask, 0)

print(f"\nMasked woreda grid created!")
print(f"Shape: {masked_woredas.shape}")

masked_woredas

In [1]:
import os
import xarray as xr
import rioxarray
import geopandas as gpd
import numpy as np
from pathlib import Path
from rasterio import features
from affine import Affine
from dask.distributed import Client

In [2]:
# Define paths
BASE_DIR = Path("..").resolve()
DATA_DIR = os.path.join("..", "data", "ethiopia")
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

### Start dask cluster

In [3]:
# Create a local dask client
client = Client(n_workers=4, threads_per_worker=2, memory_limit='2GB')
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 7.45 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:59020,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:59043,Total threads: 2
Dashboard: http://127.0.0.1:59048/status,Memory: 1.86 GiB
Nanny: tcp://127.0.0.1:59023,


2025-10-19 16:14:17,833 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:59041 (pid=9468) exceeded 95% memory budget. Restarting...
2025-10-19 16:14:17,856 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:59041' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'original-open_rasterio-883a0b5f3415f9a1bbb4eb9e330e3fc3<this-array>-0a5ca7ba0b66c347d66c4c5584225efd', ('getitem-34598af1fbeb8b59b8402d207bf8fa92', 6, 6), ('getitem-34598af1fbeb8b59b8402d207bf8fa92', 6, 2), ('open_rasterio-getitem-34598af1fbeb8b59b8402d207bf8fa92', 5, 5)} (stimulus_id='handle-worker-cleanup-1760915657.8564103')
2025-10-19 16:14:18,078 - distributed.nanny - WARNING - Restarting worker
2025-10-19 16:14:20,929 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:59034 (pid=7576) exceeded 95% memory budget. Restarting...
2025-10-19 16:14:20,939 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:59034' caused the clus

# Crop Cover Extraction for Woredas

This notebook provides functions to extract crop cover data for Ethiopian woredas using xarray and rioxarray.

For processing multiple woredas efficiently, we can rasterize the entire woreda GeoDataFrame into a grid that matches the crop cover data. This allows vectorized operations across all woredas.

In [6]:
def rasterize_woredas(woredas_gdf, resolution=0.0002694945852352859):
    """
    Rasterize woredas into a grid based on the bounds of the woredas themselves.

    Parameters
    ----------
    woredas_gdf : gpd.GeoDataFrame
        GeoDataFrame with woreda geometries
    resolution : float, optional
        Pixel resolution in degrees (default matches crop cover resolution)

    Returns
    -------
    woreda_grid : xr.DataArray
        Array where each pixel value = woreda index (0 = no woreda)
    woreda_lookup : gpd.GeoDataFrame
        Lookup table mapping woreda_id to NAME_3 and geometry
    """
    # Add a numeric ID column if it doesn't exist
    # Using the index as the woreda ID
    woredas_gdf = woredas_gdf.copy()
    woredas_gdf['woreda_id'] = woredas_gdf.index + 1  # +1 so 0 can be "no woreda"

    # Get the spatial extent from woredas bounds
    bounds = woredas_gdf.total_bounds  # (minx, miny, maxx, maxy)
    minx, miny, maxx, maxy = bounds

    # Use provided resolution
    x_res = resolution
    y_res = resolution

    # Create coordinate arrays
    # X coordinates from minx to maxx
    x_coords = np.arange(minx + x_res/2, maxx, x_res)
    # Y coordinates from maxy to miny (reversed for geospatial convention)
    y_coords = np.arange(maxy - y_res/2, miny, -y_res)

    # Calculate grid dimensions
    width = len(x_coords)
    height = len(y_coords)

    # Create affine transform
    # Affine maps from pixel coordinates to geographic coordinates
    transform = Affine.translation(x_coords[0] - x_res/2, y_coords[0] + y_res/2) * \
                Affine.scale(x_res, -y_res)

    # Prepare (geometry, value) pairs for rasterization
    shapes = ((geom, value) for geom, value in
              zip(woredas_gdf.geometry, woredas_gdf.woreda_id))

    print(f"Rasterizing {len(woredas_gdf)} woredas into {height} x {width} grid...")
    print(f"Bounds: {bounds}")
    print(f"Resolution: {resolution} degrees")

    # Rasterize using rasterio
    woreda_array = features.rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,  # Background value (no woreda)
        dtype='uint16',  # Can handle up to 65535 woredas
        all_touched=False  # Only pixels whose center is in polygon
    )

    print(f"Rasterization complete!")

    # Convert to xarray DataArray with matching coordinates
    woreda_grid = xr.DataArray(
        woreda_array,
        coords={
            'y': y_coords,
            'x': x_coords
        },
        dims=['y', 'x'],
        name='woreda_id'
    )

    # Add CRS metadata
    woreda_grid.rio.write_crs("EPSG:4326", inplace=True)

    # Add metadata
    woreda_grid.attrs['description'] = 'Woreda ID grid (0 = no woreda)'
    woreda_grid.attrs['woreda_count'] = len(woredas_gdf)
    woreda_grid.attrs['resolution'] = resolution
    woreda_grid.attrs['bounds'] = bounds.tolist()

    # Create lookup table
    woreda_lookup = woredas_gdf[['NAME_3', 'woreda_id', 'geometry']].copy()

    return woreda_grid, woreda_lookup

### Create the Woreda Grid

This is a one-time operation that creates a rasterized version of all woredas:

In [7]:
# Load woredas
woredas_fp = os.path.join(DATA_DIR, "woredas.json")
woredas_gdf = gpd.read_file(woredas_fp)

# Rasterize all woredas (using bounds from woredas_gdf itself)
woreda_grid, woreda_lookup = rasterize_woredas(woredas_gdf)

woreda_grid

Rasterizing 690 woredas into 42475 x 55499 grid...
Bounds: [33.0015  3.3988 47.9582 14.8455]
Resolution: 0.0002694945852352859 degrees
Rasterization complete!


<xarray.DataArray 'woreda_id' (y: 42475, x: 55499)> Size: 5GB
array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(42475, 55499), dtype=uint16)
Coordinates:
  * y            (y) float64 340kB 14.85 14.85 14.84 14.84 ... 3.399 3.399 3.399
  * x            (x) float64 444kB 33.0 33.0 33.0 33.0 ... 47.96 47.96 47.96
    spatial_ref  int64 8B 0
Attributes:
    description:   Woreda ID grid (0 = no woreda)
    woreda_count:  690
    resolution:    0.0002694945852352859
    bounds:        [33.0015, 3.3988, 47.9582, 14.8455]

### Save Woreda Grid to Zarr (for reuse)

In [40]:
# Save woreda grid for future use
woreda_grid.to_zarr(os.path.join(OUTPUT_DIR, "woredas.zarr"), mode='w')

# Save lookup table
woreda_lookup.to_file(os.path.join(OUTPUT_DIR, "woredas_lookup.json"), driver='GeoJSON')

print(f"✓ Woreda grid saved")
print(f"✓ Lookup table saved")

c:\Users\geo1k\.local\share\mamba\envs\geodask\Lib\site-packages\zarr\api\asynchronous.py:228: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


✓ Woreda grid saved
✓ Lookup table saved


### Save Woreda Grid to GeoTIFF

In [39]:
# Save woreda_grid to GeoTIFF with EPSG:4326
WOREDA_GRID_GEOTIFF = os.path.join(OUTPUT_DIR, "woredas.tif")

# Set the CRS to EPSG:4326
woreda_grid.rio.write_crs("EPSG:4326", inplace=True)

# Save to GeoTIFF with DEFLATE compression
woreda_grid.rio.to_raster(
    WOREDA_GRID_GEOTIFF,
    driver='GTiff',
    compress='DEFLATE',  # DEFLATE compression to reduce file size
    dtype='uint16'       # Matches the data type (can handle up to 65535 woredas)
)

print(f"✓ Woreda grid saved to {WOREDA_GRID_GEOTIFF}")

✓ Woreda grid saved to ..\data\ethiopia\output\woredas.tif


In [4]:
# Load crop cover
crop_cover_fp = os.path.join(DATA_DIR, "crop_cover", "crop_cover.vrt")
crop_cover = rioxarray.open_rasterio(crop_cover_fp, chunks="auto")
if 'band' in crop_cover.dims:
    crop_cover = crop_cover.squeeze('band', drop=True)

crop_cover

<xarray.DataArray (y: 74482, x: 74222)> Size: 6GB
dask.array<getitem, shape=(74482, 74222), dtype=uint8, chunksize=(11520, 11520), chunktype=numpy.ndarray>
Coordinates:
  * x            (x) float64 594kB 30.0 30.0 30.0 30.0 ... 50.0 50.0 50.0 50.0
  * y            (y) float64 596kB 20.07 20.07 20.07 ... -0.0006737 -0.0009432
    spatial_ref  int64 8B 0
Attributes:
    STATISTICS_APPROXIMATE:    YES
    STATISTICS_MAXIMUM:        2
    STATISTICS_MEAN:           0.98848051415196
    STATISTICS_MINIMUM:        0
    STATISTICS_STDDEV:         0.55342025644358
    STATISTICS_VALID_PERCENT:  100
    scale_factor:              1.0
    add_offset:                0.0

In [50]:
client.shutdown()